# Zerlaut parameter exploration

## Initialization

**Environment**

In [ ]:
from dotenv import dotenv_values, find_dotenv
from pathlib import Path
import os, sys

# Busqueda de ficheros dotenv
defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

# Actualización de varaibles de entorno
settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }

os.environ.update(settings)

# Librerías auxiliares
import_dir = Path(settings.get('IMPORT_DIR') or env_dir).expanduser()
if not import_dir.is_absolute():
    import_dir = env_dir / import_dir
sys.path.insert(0, str(import_dir.resolve()))

**Modules**

In [ ]:
import json5 as json
from copy import deepcopy

In [ ]:
from aux.environment import resolve_directory
from aux.parameters import bounds_from_default, filter_nested_dict, add_beta_config_safe

**Console**

In [ ]:
from rich.console import Console
pprint = Console(force_terminal=True, force_jupyter=False, width=140).print
pprint(f'The console is "ready" to print messages with rich formatting.')

**Set main directory paths**

In [ ]:
WORK_DIR = resolve_directory(settings['WORK_DIR'], env_dir)

os.makedirs(WORK_DIR, exist_ok=True)
pprint(f'Working directory: {WORK_DIR}')

## Auxiliar Functions

In [ ]:
def remove_unsampled_parameters(params, names) -> dict:
    result = deepcopy(params)
    for key in names:
        result.pop(key, None) # Evita el error si no se encuentra
    return result

In [ ]:
def count_parameters(params):
    return sum(
        len(v['default']) if isinstance(v['default'], list) else 1 
        for v in params.values()
    )

## Parameter sampling


In [ ]:
CONFIG_PATH = WORK_DIR / 'config' / 'zerlaut' / 'params_raw.json'
pprint(f'Parameter configuration: {CONFIG_PATH}')

In [ ]:
with CONFIG_PATH.open(encoding='utf-8') as config_file:
    PARAMS = json.load(config_file)

SAMPLED_PARAMS, FIX_PARAMS = deepcopy(PARAMS), {}
SAMPLED_PARAMS = filter_nested_dict(SAMPLED_PARAMS, ['min', 'max', 'default'])

pprint(f'Loaded {count_parameters(PARAMS)}', str(list(PARAMS.keys())))

### Common processing

1. **Remove unsampled parameters**

In [ ]:
SAMPLED_PARAMS = remove_unsampled_parameters(SAMPLED_PARAMS, [
    "external_input_ex_ex", "external_input_ex_in", "external_input_in_ex", "external_input_in_in" #,S_i
])

2. **Add undefined bounds**

In [ ]:
SAMPLED_PARAMS = bounds_from_default(SAMPLED_PARAMS, names=["P_e", "P_i"], margin=0.10)

for param in (SAMPLED_PARAMS['P_e'], SAMPLED_PARAMS['P_i']):
    p = param['default'][0]
    param['min'][0], param['max'][0] = p - 0.05 * p, p + 0.05 * p

# NOTE: Puede merecer la pena usar una distribución beta para el muetreo


3. **Adjust noise**

In [ ]:
## Reduce noise
SAMPLED_PARAMS['tau_OU'] |= {'min': 0.05, 'max': 4, 'default': 2}
SAMPLED_PARAMS['weight_noise'] |= {'min': 0.00001, 'max': 0.0004, 'default': 0.1}

# NOTE: Puede merecer la pena usar una distribución beta para el muetreo

4. **Cell**

In [ ]:
# SAMPLED_PARAMS['g_L'] |= {'min': 1.0, 'max': 50.0, 'default': 10.0}
# SAMPLED_PARAMS['C_m'] |= {'min': 50, 'max': 250, 'default': 200}


5. **Adaptation**

TODO;

Son inestables si crecen los 2!

Exploring parameters: ['E_e', 'E_i']
Default: [0.0, -80.0]
Min: [-20.0, -100.0]
Max: [20.0, -60.0]

Insestables para valores altos de Q_e con valores bajos de Q_i

Exploring parameters: ['E_e', 'E_i']
Default: [0.0, -80.0]
Min: [-20.0, -100.0]
Max: [20.0, -60.0]

No dejar subir Q_e de dos!

In [ ]:
# SAMPLED_PARAMS["Q_e"] |= {'min': 0.5, 'max': 2.0, 'default': 1.0}

### Parameter configuration 1

* Add beta distiribution to all

In [ ]:
PARAM_CONFIG1 = deepcopy(SAMPLED_PARAMS)
FIX_PARAMS1 = deepcopy(FIX_PARAMS)

In [ ]:
PARAM_CONFIG1 = add_beta_config_safe(PARAM_CONFIG1, lambda_=2) # names=["P_e", "P_i", "weight_noise", "tau_OU"], 

In [ ]:
pprint(f'Loaded {count_parameters(PARAM_CONFIG1)}', str(list(PARAM_CONFIG1.keys())))

## Stimulus configuration


Ver 1c

In [ ]:
stimulus_config = {
    "Random smooth transitions": {
        "external_input_ex_ex": {
            "random_smooth": {
                "baseline_khz": 0.005, "onset_ms": 500.0,
                "min_duration_ms": 100.0, "max_duration_ms": 250.0,
                "min_amplitude_khz": 0.005, "max_amplitude_khz": 0.04,
                # Estos valores se tendran que generar aleatoriamente.
                # "seed": 42, 
            }
        },
        "external_input_ex_in": { "external_input_ex_ex": 1} # Valor a multiplicar por S_i            
    },
    "Random smooth transitions + noise 1": {
        "external_input_ex_ex": {
            "random_smooth": {
                "baseline_khz": 0.005, "onset_ms": 500.0,
                "min_duration_ms": 100.0, "max_duration_ms": 250.0,
                "min_amplitude_khz": 0.005, "max_amplitude_khz": 0.04,
                "mean_khz": 0.0, "std_khz": 0.001,
                # Estos valores se tendran que generar aleatoriamente.
                # "seed": 42, 
                # "noise_seed": 7
            }
        },
        "external_input_ex_in": { "external_input_ex_ex": 1} # Valor a multiplicar por S_i            
    },
}

stimulus_prob = [0.2, 0.4, 0.4]
    

